In [1]:
import pandas as pd
import numpy as np

In [3]:
file_path = r'memory_optimize.csv'

df_original  = pd.read_csv(file_path, encoding='utf-8', parse_dates=['注册时间'])

df_original

,用户ID,姓名,性别,年龄,城市,会员等级,消费金额(元),下单次数,注册时间,是否付费
0,1001,张三,男,25,北京,VIP3,2999.0,5,2024-01-10,是
1,1002,李四,女,32,上海,VIP2,199.5,3,2024-02-15,是
2,1003,王五,男,28,广州,VIP1,318.8,2,2024-01-20,否
3,1004,赵六,女,40,深圳,VIP3,99.0,1,2024-03-05,否
4,1005,钱七,男,35,北京,VIP2,2999.0,4,2024-02-20,是
5,1006,孙八,女,29,上海,VIP1,299.9,2,2024-03-10,是
6,1007,周九,男,45,广州,VIP3,129.8,1,2024-01-25,否
7,1008,吴十,女,27,深圳,VIP2,799.5,3,2024-04-05,是
8,1009,郑十一,男,33,北京,VIP1,599.0,2,2024-02-18,否
9,1010,冯十二,女,38,上海,VIP3,1999.9,5,2024-03-22,是


In [4]:
df_original.dtypes

用户ID                int64
姓名                    str
性别                    str
年龄                  int64
城市                    str
会员等级                  str
消费金额(元)           float64
下单次数                int64
注册时间       datetime64[us]
是否付费                  str
dtype: object

In [5]:
original_memory = df_original.memory_usage(deep=True).sum() / 1024 / 1024

print(f"原始内存占用：{original_memory:.4f} MB")

原始内存占用：0.0039 MB


In [6]:
print("各字段内存占用（MB）：\n", (df_original.memory_usage(deep=True)/1024/1024).round(4))

各字段内存占用（MB）：
 Index      0.0001
用户ID       0.0001
姓名         0.0007
性别         0.0007
年龄         0.0001
城市         0.0007
会员等级       0.0005
消费金额(元)    0.0001
下单次数       0.0001
注册时间       0.0001
是否付费       0.0007
dtype: float64


In [8]:
df_original.dtypes

用户ID                int64
姓名                    str
性别                    str
年龄                  int64
城市                    str
会员等级                  str
消费金额(元)           float64
下单次数                int64
注册时间       datetime64[us]
是否付费                  str
dtype: object

In [7]:
df_optimized = df_original.copy()

df_optimized

,用户ID,姓名,性别,年龄,城市,会员等级,消费金额(元),下单次数,注册时间,是否付费
0,1001,张三,男,25,北京,VIP3,2999.0,5,2024-01-10,是
1,1002,李四,女,32,上海,VIP2,199.5,3,2024-02-15,是
2,1003,王五,男,28,广州,VIP1,318.8,2,2024-01-20,否
3,1004,赵六,女,40,深圳,VIP3,99.0,1,2024-03-05,否
4,1005,钱七,男,35,北京,VIP2,2999.0,4,2024-02-20,是
5,1006,孙八,女,29,上海,VIP1,299.9,2,2024-03-10,是
6,1007,周九,男,45,广州,VIP3,129.8,1,2024-01-25,否
7,1008,吴十,女,27,深圳,VIP2,799.5,3,2024-04-05,是
8,1009,郑十一,男,33,北京,VIP1,599.0,2,2024-02-18,否
9,1010,冯十二,女,38,上海,VIP3,1999.9,5,2024-03-22,是


In [10]:
df_optimized['年龄'] = df_optimized['年龄'].astype('int8')
df_optimized['下单次数'] = df_optimized['下单次数'].astype('int8')
df_optimized['用户ID'] = df_optimized['用户ID'].astype('int16')

df_optimized['消费金额(元)'] = df_optimized['消费金额(元)'].astype('float32')

for col in ['性别', '城市', '会员等级']:
    df_optimized[col] = df_optimized[col].astype('category')

df_optimized['是否付费'] = df_optimized['是否付费'].map({'是': True, '否': False}).astype('bool')

In [11]:
df_optimized.dtypes

用户ID                int16
姓名                    str
性别               category
年龄                   int8
城市               category
会员等级             category
消费金额(元)           float32
下单次数                 int8
注册时间       datetime64[us]
是否付费                 bool
dtype: object

In [12]:
optimized_memory = df_optimized.memory_usage(deep=True).sum() / 1024 / 1024

print(f"优化后内存占用：{optimized_memory:.4f} MB")

优化后内存占用：0.0016 MB


In [13]:
memory_save = (original_memory - optimized_memory) / original_memory * 100
print(f"内存节省比例：{memory_save:.2f}%")

内存节省比例：57.48%


In [14]:
print(f"数据行数是否一致：{len(df_original) == len(df_optimized)}")

数据行数是否一致：True


In [15]:
print(f"消费金额均值是否一致：{np.isclose(df_original['消费金额(元)'].mean(), df_optimized['消费金额(元)'].mean())}")

消费金额均值是否一致：True


In [16]:
dtype_spec = {
    '用户ID': 'int16',
    '年龄': 'int8',
    '性别': 'category',
    '城市': 'category',
    '会员等级': 'category',
    '消费金额(元)': 'float32',
    '下单次数': 'int8',
    '是否付费': 'object'
}

df_read_opt = pd.read_csv(
    file_path,
    encoding='utf-8',
    parse_dates=['注册时间'],
    dtype=dtype_spec
)

df_read_opt

,用户ID,姓名,性别,年龄,城市,会员等级,消费金额(元),下单次数,注册时间,是否付费
0,1001,张三,男,25,北京,VIP3,2999.000000,5,2024-01-10,是
1,1002,李四,女,32,上海,VIP2,199.500000,3,2024-02-15,是
2,1003,王五,男,28,广州,VIP1,318.799988,2,2024-01-20,否
3,1004,赵六,女,40,深圳,VIP3,99.000000,1,2024-03-05,否
4,1005,钱七,男,35,北京,VIP2,2999.000000,4,2024-02-20,是
5,1006,孙八,女,29,上海,VIP1,299.899994,2,2024-03-10,是
6,1007,周九,男,45,广州,VIP3,129.800003,1,2024-01-25,否
7,1008,吴十,女,27,深圳,VIP2,799.500000,3,2024-04-05,是
8,1009,郑十一,男,33,北京,VIP1,599.000000,2,2024-02-18,否
9,1010,冯十二,女,38,上海,VIP3,1999.900024,5,2024-03-22,是


In [17]:
df_read_opt['是否付费'] = df_read_opt['是否付费'].map({'是': True, '否': False}).astype('bool')

In [18]:
read_opt_memory = df_read_opt.memory_usage(deep=True).sum() / 1024 / 1024
print(f"\n4. 读取时指定类型内存占用：{read_opt_memory:.4f} MB")


4. 读取时指定类型内存占用：0.0020 MB


In [19]:
optimize_table = pd.DataFrame(
    {
        '原始类型': ['int64', 'float64', 'object(低基数)', 'object(是/否)'],
        '优化类型': ['int8/int16/int32', 'float32', 'category', 'bool'],
        '内存节省': ['~75%~93%', '~50%', '~50%~90%', '~90%']
    }
)
print(optimize_table)

          原始类型              优化类型      内存节省
0        int64  int8/int16/int32  ~75%~93%
1      float64           float32      ~50%
2  object(低基数)          category  ~50%~90%
3  object(是/否)              bool      ~90%
